# PTB-XL ECG Diagnosis — Ablation Study

Reproduces part of the benchmark from:
> Nonaka, K., & Seita, D. (2021). *In-depth Benchmarking of Deep Neural Network Architectures for ECG Diagnosis.* Proceedings of Machine Learning Research, 149, 414-424.

This notebook runs an ablation study on the **real PTB-XL dataset** using PyHealth's pipeline:
- `PTBXLDataset` → `PTBXLDiagnosis` / `PTBXLMulticlassDiagnosis` → MLP / CNN / Transformer → Trainer

**Ablation dimensions:**
1. Task type — multilabel vs multiclass
2. Model architecture — MLP, CNN, Transformer
3. Hidden dim — 32, 64, 128

**Authors:** Ankita Jain (ankitaj3@illinois.edu), Manish Singh (manishs4@illinois.edu)

## 0. Setup & Install Dependencies

In [ ]:
# Install PyHealth and wfdb (run once on Colab)
!pip install -q pyhealth wfdb requests

## 1. Download PTB-XL Dataset

Auto-downloads from PhysioNet with resume support. Re-run this cell if download is interrupted.

In [ ]:
import os
import time
import zipfile
from pathlib import Path

import requests

PTBXL_URL = "https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
PTBXL_ZIP = "ptb-xl-1.0.3.zip"
PTBXL_EXTRACTED_DIR = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
DATA_DIR = "./data"


def download_ptbxl(dest_dir=DATA_DIR, max_retries=5):
    """Download and extract PTB-XL with resume-on-failure."""
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    extracted_root = dest / PTBXL_EXTRACTED_DIR

    if (extracted_root / "ptbxl_database.csv").exists():
        print(f"PTB-XL already available at {extracted_root}")
        return str(extracted_root)

    zip_path = dest / PTBXL_ZIP
    print(f"Downloading PTB-XL (~1.8 GB) ...")

    for attempt in range(1, max_retries + 1):
        try:
            downloaded = zip_path.stat().st_size if zip_path.exists() else 0
            headers = {"Range": f"bytes={downloaded}-"} if downloaded > 0 else {}
            if downloaded > 0:
                print(f"Resuming from {downloaded / 1e6:.1f} MB (attempt {attempt})")

            resp = requests.get(PTBXL_URL, headers=headers, stream=True, timeout=60)
            if resp.status_code == 200:
                downloaded, mode = 0, "wb"
            elif resp.status_code == 206:
                mode = "ab"
            elif resp.status_code == 416:
                break
            else:
                resp.raise_for_status()

            total = int(resp.headers.get("content-length", 0)) + downloaded
            with open(zip_path, mode) as f:
                for chunk in resp.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        pct = downloaded / total * 100 if total else 0
                        print(f"\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.1f}%)", end="", flush=True)
            print("\nDownload complete.")
            break
        except (requests.RequestException, IOError) as e:
            print(f"\nError (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                time.sleep(min(2 ** attempt, 30))
            else:
                raise RuntimeError(f"Download failed after {max_retries} attempts.")

    print("Extracting ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest)
    print(f"Done: {extracted_root}")
    return str(extracted_root)


ROOT = download_ptbxl()

## 2. Load Dataset & Generate Task Samples

In [ ]:
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)

from pyhealth.datasets.ptbxl import PTBXLDataset
from pyhealth.tasks.ptbxl_diagnosis import (
    PTBXLDiagnosis,
    PTBXLMulticlassDiagnosis,
    SUPERCLASSES,
)

dataset = PTBXLDataset(root=ROOT, sampling_rate=100)

# Collect task samples via iter_patients
ml_samples, mc_samples = [], []
ml_task, mc_task = PTBXLDiagnosis(), PTBXLMulticlassDiagnosis()

for patient in dataset.iter_patients():
    ml_samples.extend(ml_task(patient))
    mc_samples.extend(mc_task(patient))

print(f"Multilabel samples : {len(ml_samples)}")
print(f"Multiclass samples : {len(mc_samples)}")

## 3. Ablation 1 — Task Type: Label Distribution

In [ ]:
from collections import Counter

ml_dist = Counter(l for s in ml_samples for l in s["labels"])
mc_dist = Counter(s["label"] for s in mc_samples)

print("Multilabel distribution:", dict(sorted(ml_dist.items())))
print("Multiclass distribution:", dict(sorted(mc_dist.items())))

## 4. Load Real ECG Signals & Build SampleDataset

In [ ]:
import wfdb
from pyhealth.datasets import create_sample_dataset, get_dataloader, split_by_sample


def load_ecg_signal(root, signal_file):
    """Load 12-lead ECG via wfdb, return flattened list or None."""
    try:
        signal, _ = wfdb.rdsamp(os.path.join(root, signal_file))
        return signal.flatten().tolist()
    except Exception:
        return None


def build_dataset(task_samples, root, task_type="multiclass"):
    """Load real ECG signals and build a PyHealth SampleDataset."""
    feature_samples, skipped = [], 0
    total = len(task_samples)

    for i, s in enumerate(task_samples):
        if (i + 1) % 2000 == 0:
            print(f"\r  Loading signals: {i + 1}/{total}", end="", flush=True)
        signal = load_ecg_signal(root, s["signal_file"])
        if signal is None:
            skipped += 1
            continue
        sample = {
            "patient_id": s["patient_id"],
            "visit_id": str(s["record_id"]),
            "ecg_signal": signal,
        }
        if task_type == "multilabel":
            sample["labels"] = s["labels"]
        else:
            sample["label"] = s["label"]
        feature_samples.append(sample)

    print(f"\r  Loaded {len(feature_samples)} signals ({skipped} skipped)")

    label_key = "labels" if task_type == "multilabel" else "label"
    label_mode = "multilabel" if task_type == "multilabel" else "multiclass"

    return create_sample_dataset(
        samples=feature_samples,
        input_schema={"ecg_signal": "sequence"},
        output_schema={label_key: label_mode},
        dataset_name="ptbxl_real",
    )

In [ ]:
print("Building multiclass dataset ...")
mc_ds = build_dataset(mc_samples, ROOT, task_type="multiclass")

print("Building multilabel dataset ...")
ml_ds = build_dataset(ml_samples, ROOT, task_type="multilabel")

## 5. Ablation 2 — Model Architecture × Hidden Dim

We train MLP, CNN, and Transformer across hidden dims {32, 64, 128}.

In [ ]:
from pyhealth.models import MLP, CNN, Transformer
from pyhealth.trainer import Trainer

EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
HIDDEN_DIMS = [32, 64, 128]
MODEL_CONFIGS = [(MLP, "MLP"), (CNN, "CNN"), (Transformer, "Transformer")]


def run_ablation(sample_ds, model_cls, hidden_dim, metrics_list):
    """Train + evaluate one model config. Returns val/test score dicts."""
    train_ds, val_ds, test_ds = split_by_sample(sample_ds, [0.7, 0.15, 0.15])
    train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = get_dataloader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = get_dataloader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    if model_cls is MLP:
        model = model_cls(dataset=sample_ds, hidden_dim=hidden_dim, n_layers=2)
    elif model_cls is CNN:
        model = model_cls(dataset=sample_ds, hidden_dim=hidden_dim, num_layers=2)
    elif model_cls is Transformer:
        model = model_cls(dataset=sample_ds, embedding_dim=hidden_dim, num_layers=2)

    trainer = Trainer(model=model, metrics=metrics_list, enable_logging=False)
    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=EPOCHS,
        optimizer_params={"lr": LR},
    )
    return {
        "val": trainer.evaluate(val_loader),
        "test": trainer.evaluate(test_loader),
    }

### 5a. Multiclass Ablation (F1, Accuracy)

In [ ]:
mc_metrics = ["f1_weighted", "accuracy"]
mc_results = []

print(f"{'Model':<14} {'hidden_dim':<12} {'val_f1':<10} {'test_f1':<10} {'test_acc':<10} {'test_loss'}")
print("-" * 68)

for model_cls, model_name in MODEL_CONFIGS:
    for hd in HIDDEN_DIMS:
        try:
            r = run_ablation(mc_ds, model_cls, hd, mc_metrics)
            v, t = r["val"], r["test"]
            print(
                f"{model_name:<14} {hd:<12} "
                f"{v.get('f1_weighted', 0):<10.4f} "
                f"{t.get('f1_weighted', 0):<10.4f} "
                f"{t.get('accuracy', 0):<10.4f} "
                f"{t.get('loss', 0):.4f}"
            )
            mc_results.append({"model": model_name, "hidden_dim": hd, **t})
        except Exception as e:
            print(f"{model_name:<14} {hd:<12} FAILED: {e}")

### 5b. Multilabel Ablation (ROC-AUC, F1)

In [ ]:
ml_metrics = ["roc_auc_samples", "f1_weighted"]
ml_results = []

print(f"{'Model':<14} {'hidden_dim':<12} {'val_auroc':<12} {'test_auroc':<12} {'test_f1':<10} {'test_loss'}")
print("-" * 72)

for model_cls, model_name in MODEL_CONFIGS:
    for hd in HIDDEN_DIMS:
        try:
            r = run_ablation(ml_ds, model_cls, hd, ml_metrics)
            v, t = r["val"], r["test"]
            print(
                f"{model_name:<14} {hd:<12} "
                f"{v.get('roc_auc_samples', 0):<12.4f} "
                f"{t.get('roc_auc_samples', 0):<12.4f} "
                f"{t.get('f1_weighted', 0):<10.4f} "
                f"{t.get('loss', 0):.4f}"
            )
            ml_results.append({"model": model_name, "hidden_dim": hd, **t})
        except Exception as e:
            print(f"{model_name:<14} {hd:<12} FAILED: {e}")

## 6. Ablation 3 — SCP Code Coverage

In [ ]:
from pyhealth.tasks.ptbxl_diagnosis import SCP_TO_SUPER

print(f"Total mapped SCP codes : {len(SCP_TO_SUPER)}")
print(f"Superclasses covered   : {sorted(set(SCP_TO_SUPER.values()))}")

## 7. Summary & Comparison with Paper

**Reference:** Nonaka & Seita (2021) reported on real PTB-XL:
- Multilabel ROC-AUC ~0.93 (ResNet)
- Multiclass F1 ~0.82 (ResNet)

Our results use MLP/CNN/Transformer (not ResNet) with 10 epochs. To close the gap:
1. More epochs (50+)
2. Larger hidden dims (256+)
3. Learning rate scheduling
4. 500 Hz signals for higher resolution

## 8. Ablation Ideas Considered but Not Used

We asked an LLM to suggest additional ablation dimensions. Here are three ideas and why we chose not to pursue them:

### Idea 1: Signal Preprocessing — Bandpass Filtering vs Raw
**LLM suggestion:** Apply bandpass filtering (0.5–40 Hz) to remove baseline wander and high-frequency noise before feeding signals to the model, and compare against raw signals.

**Why not used:** PTB-XL signals are already preprocessed by PhysioNet with standard clinical filtering. Adding another filtering step would deviate from the Nonaka & Seita (2021) benchmark setup, making our results non-comparable. The paper uses raw PTB-XL signals directly.

### Idea 2: Lead Subset Ablation — 12-lead vs 6-lead vs Single-lead
**LLM suggestion:** Compare diagnostic performance using all 12 leads vs limb leads only (6) vs a single lead (Lead II), since some clinical settings only have access to reduced lead sets.

**Why not used:** This would require modifying the signal loading pipeline to select specific lead subsets, and the Nonaka & Seita benchmark exclusively uses 12-lead input. While clinically interesting, it's outside the scope of reproducing the paper's results and would add significant implementation complexity to the PyHealth task classes.

### Idea 3: Patient-level vs Record-level Splitting
**LLM suggestion:** Compare model performance when splitting by patient ID (no patient appears in both train and test) vs random record-level splitting, to measure data leakage effects.

**Why not used:** While methodologically important, the Nonaka & Seita paper uses the official PTB-XL train/val/test splits (strat_fold column). Our current pipeline uses `split_by_sample` for simplicity. Implementing the official fold-based split would be a valuable future extension but was deprioritized in favor of the model architecture and hidden dim ablations which directly reproduce the paper's core experiments.

In [ ]:
# Cleanup
mc_ds.close()
ml_ds.close()
print("Done.")